In [1]:
# 05 - The benchmark of our models -> ONNX (for Service 3: audio-engine)
#
# The project: the audio engine runs the benchmark of our models (TP3 SVM, TP4 decision tree, TP7 CNN) with
# ONNX Runtime, and the official answer is the model with the best RECALL (it misses the fewest sick hearts).
# ONNX = 1 small file per model, that the container runs fast without TensorFlow or scikit-learn.
#
# 1. MLflow: the best run of each model, sorted by recall (the first one = the best = the official model)
# 2. export each model to ONNX (the best one = cardio_model.onnx) and check that ONNX gives the same answers
# 3. benchmark.json = the list of our models and their scores, for the audio engine
# 4. MLflow: the Model Registry (cardio_model) + the ONNX files in the runs

In [2]:
# The first time only: the tools to export to ONNX and to run an ONNX model. Then click "Restart" (the kernel).
%pip install tf2onnx==1.17.0 onnx==1.23.2 onnxruntime==1.23.2 skl2onnx==1.20.0

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'D:\MLOps_Project\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"    # hide the TensorFlow info messages

import json
import pickle
import numpy as np
import pandas as pd
import IPython.display as ipd

import mlflow
import mlflow.keras
import mlflow.sklearn
import onnxruntime as ort
from skl2onnx import to_onnx    # scikit-learn model -> ONNX (for the SVM and the tree)

In [4]:
# my folders
project_folder = "D:/MLOps_Project/"
engine_folder = "D:/MLOps_Project/audio-engine/"    # the folder of Service 3 (its Dockerfile is there)
os.makedirs(engine_folder, exist_ok=True)


def load_data(file_name):
    with open(file_name, "rb") as f:
        return pickle.load(f)


# MLflow: the container must be running (docker start mlflow-server)
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("audio_benchmark")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1790783986045, experiment_id='1', last_update_time=1790783986045, lifecycle_stage='active', name='audio_benchmark', tags={}>

In [5]:
# ==================== 1. The benchmark: the best run of each model, from MLflow ====================
runs = mlflow.search_runs(experiment_names=["audio_benchmark"], order_by=["metrics.mean_recall DESC"])

# we keep the runs that have a saved model, with features that the audio engine knows how to make
# (the same code as in the notebooks: TP4 = MFCC, TP3 = Mel spectrogram flattened, TP7 = Mel spectrogram image)
known_features = ["MFCC mean + std (26)", "mel spectrogram flatten (30 bands)", "mel spectrogram 40 x 157 (5 s, 2000 Hz)"]
has_model = runs["tags.mlflow.log-model.history"].notna()
runs = runs[has_model & runs["params.features"].isin(known_features)]

# the runs are sorted by recall -> the first run of each model = its best run
best_runs = runs.drop_duplicates("params.model")
columns = ["tags.mlflow.runName", "params.model", "metrics.mean_recall", "metrics.mean_precision",
           "metrics.mean_f1", "metrics.mean_accuracy"]
ipd.display(best_runs[columns].round(3))
print("The best model (the official answer of the audio engine):", best_runs.iloc[0]["params.model"])
if "DecisionTree" not in list(best_runs["params.model"]):
    print("-> no decision tree here: run the tree notebook (03) again, so its run is in audio_benchmark")

,tags.mlflow.runName,params.model,metrics.mean_recall,metrics.mean_precision,metrics.mean_f1,metrics.mean_accuracy
0,cnn,CNN,0.925,0.510,0.655,0.778
2,svm_rbf_5s,SVM,0.829,0.521,0.639,0.791
6,decision_tree_gini_depth10,DecisionTree,0.718,0.559,0.629,0.810


The best model (the official answer of the audio engine): CNN


In [6]:
# ==================== 2. Export each model to ONNX, and check that ONNX gives the same answers ====================
# the features saved by each notebook (the same wavs), to compare each model with its ONNX version
saved_features = {
    "DecisionTree": load_data(project_folder + "mfcc_features_tree_a.data"),
    "SVM": load_data(project_folder + "mel_features_svm_a.data"),
    "CNN": load_data(project_folder + "cnn_spectrograms.data"),
}

models = []    # the information of each model, for benchmark.json
for i in range(len(best_runs)):
    run = best_runs.iloc[i]
    name = run["params.model"]
    X = np.array(saved_features[name]).astype("float32")    # float32: the type ONNX wants

    # the best model (i = 0) gets the name asked in the project: cardio_model.onnx
    if i == 0:
        file_name = "cardio_model.onnx"
    else:
        file_name = name.lower() + ".onnx"

    if name == "CNN":
        X = ((X + 80) / 80).reshape(len(X), 40, 157, 1)    # prepared like in TP7
        model = mlflow.keras.load_model("runs:/" + run["run_id"] + "/model")
        y_model = (model.predict(X, verbose=0).ravel() > 0.5).astype(int)
        model.export(engine_folder + file_name, format="onnx")    # Keras -> ONNX (it uses tf2onnx)
    else:
        model = mlflow.sklearn.load_model("runs:/" + run["run_id"] + "/model")
        y_model = model.predict(X)
        onnx_model = to_onnx(model, X[:1])                        # scikit-learn -> ONNX (skl2onnx)
        with open(engine_folder + file_name, "wb") as f:
            f.write(onnx_model.SerializeToString())

    # check with ONNX Runtime (what the container uses): the same decision for every wav?
    session = ort.InferenceSession(engine_folder + file_name)
    outputs = session.run(None, {session.get_inputs()[0].name: X})
    if name == "CNN":
        y_onnx = (outputs[0].ravel() > 0.5).astype(int)    # the CNN gives the probability of "abnormal"
    else:
        y_onnx = outputs[0]                                 # the SVM and the tree give the class (0 or 1)
    print(name, "->", file_name, "| same decision for", np.sum(y_model == y_onnx), "of", len(X), "wavs")

    models.append({
        "model": name,
        "file": file_name,
        "features": run["params.features"],
        "seconds": 5 if name == "CNN" else int(run["params.seconds"]),    # the CNN of TP7 always uses 5 s
        "recall": round(float(run["metrics.mean_recall"]), 3),
        "precision": round(float(run["metrics.mean_precision"]), 3),
        "f1": round(float(run["metrics.mean_f1"]), 3),
        "accuracy": round(float(run["metrics.mean_accuracy"]), 3),
        "mlflow_run_id": run["run_id"],
    })

Saved artifact at 'D:/MLOps_Project/audio-engine/cardio_model.onnx'.


CNN -> cardio_model.onnx | same decision for 3681 of 3681 wavs


SVM -> svm.onnx | same decision for 3681 of 3681 wavs


DecisionTree -> decisiontree.onnx | same decision for 3681 of 3681 wavs


In [7]:
# ==================== 3. benchmark.json: the models of the audio engine, the best one first ====================
benchmark = {"best_model": models[0]["model"], "models": models}
with open(engine_folder + "benchmark.json", "w") as f:
    json.dump(benchmark, f, indent=4)
ipd.display(pd.DataFrame(models).drop(columns=["mlflow_run_id"]))

,model,file,features,seconds,recall,precision,f1,accuracy
0,CNN,cardio_model.onnx,"mel spectrogram 40 x 157 (5 s, 2000 Hz)",5,0.925,0.510,0.655,0.778
1,SVM,svm.onnx,mel spectrogram flatten (30 bands),5,0.829,0.521,0.639,0.791
2,DecisionTree,decisiontree.onnx,MFCC mean + std (26),5,0.718,0.559,0.629,0.810


In [8]:
# ==================== 4. MLflow ====================
# the Model Registry = the shelf of our "official" models: the best model is saved as "cardio_model" (version 1, 2, ...)
mlflow.register_model("runs:/" + models[0]["mlflow_run_id"] + "/model", "cardio_model")

# each ONNX file goes in the run of its model (next to the original model), and benchmark.json with the best one
for m in models:
    with mlflow.start_run(run_id=m["mlflow_run_id"]):
        mlflow.log_artifact(engine_folder + m["file"])
with mlflow.start_run(run_id=models[0]["mlflow_run_id"]):
    mlflow.log_artifact(engine_folder + "benchmark.json")

Registered model 'cardio_model' already exists. Creating a new version of this model...


2026/11/22 18:03:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: cardio_model, version 3


Created version '3' of model 'cardio_model'.


🏃 View run cnn at: http://localhost:5000/#/experiments/1/runs/6516808df9834234a5f069a2f472ccc1
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run svm_rbf_5s at: http://localhost:5000/#/experiments/1/runs/b3611558182840e399552317c9a2b606
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run decision_tree_gini_depth10 at: http://localhost:5000/#/experiments/1/runs/c07a9e71ed3b4230977b362743c6c928
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run cnn at: http://localhost:5000/#/experiments/1/runs/6516808df9834234a5f069a2f472ccc1
🧪 View experiment at: http://localhost:5000/#/experiments/1
